XAI Analysis (LIME and Integrated Gradients)
This consolidated script loads a mock Hugging Face model, defines necessary prediction and visualization utilities, and then performs LIME and Integrated Gradients (IG) attribution

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
import numpy as np
import warnings
import os
import sys

# New imports for Integrated Gradients (using Captum)
from captum.attr import IntegratedGradients

# --- Custom Visualization Function (Replaces Captum's visualize_text for VS Code compatibility) ---
# NOTE: This function generates the HTML string. To view it, you must save it to a file
# and open that file in a web browser, or use a notebook/interactive editor.

def plot_text_heatmap(tokens, attributions, output_filename="ig_heatmap_output.html"):
    """
    Generates an HTML representation of the text with attribution heatmap
    and saves it to a file. Positive scores (driving prediction) are green,
    negative scores are red.
    """
    if not tokens or not attributions:
        print("No tokens or attributions to display.")
        return

    # Normalize attributions for color intensity (scale to -1 to 1)
    max_abs = np.max(np.abs(attributions))

    if max_abs == 0:
        norm_attributions = np.zeros_like(attributions)
    else:
        norm_attributions = attributions / max_abs

    html_output = '<html><body>'
    html_output += '<h2 style="font-family: Arial;">Integrated Gradients Heatmap</h2>'
    html_output += '<div style="line-height: 1.8; font-size: 16px; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background-color: #f9f9f9; max-width: 900px; font-family: Arial;">'

    for token, score in zip(tokens, norm_attributions):
        # Determine color and intensity
        intensity = min(1.0, abs(score) * 2.5) # Amplify intensity slightly

        if score > 0:
            # Green for positive contribution (to the predicted class)
            style = f'background-color: rgba(0, 128, 0, {intensity:.2f}); color: black; padding: 2px 3px; border-radius: 3px; margin: 1px; display: inline-block;'
        elif score < 0:
            # Red for negative contribution (against the predicted class)
            style = f'background-color: rgba(255, 0, 0, {intensity:.2f}); color: black; padding: 2px 3px; border-radius: 3px; margin: 1px; display: inline-block;'
        else:
            # Neutral/Zero contribution
            style = 'background-color: transparent; color: #333; padding: 2px 3px; margin: 1px; display: inline-block;'

        # Replace subword tokenization prefix (like '##')
        display_token = token.replace('##', '')

        html_output += f'<span style="{style}">{display_token}</span>'

    html_output += '</div></body></html>'
    
    # Save the HTML output to a file
    try:
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write(html_output)
        print(f"\n[INFO] HTML Heatmap saved to: {os.path.abspath(output_filename)}")
        return output_filename
    except Exception as e:
        print(f"[ERROR] Could not save HTML file: {e}")
        return None


# --- Suppress Warnings ---
warnings.filterwarnings('ignore')

# --- 1. Configuration and Model Loading (Mocked for environment setup) ---
# NOTE: Replace 'google/electra-small-discriminator' with your actual saved model path.
MODEL_NAME = 'google/electra-small-discriminator'
NUM_LABELS = 3
CLASS_NAMES = ['HAM', 'SPAM', 'AI-SPAM'] # Ensure this matches your final class mapping
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("--- Initializing Model and Tokenizer ---")
try:
    # Load the base model and tokenizer for demonstration
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(DEVICE)
    model.eval()
    print(f"Model and Tokenizer loaded successfully on {DEVICE}.")
except Exception as e:
    print(f"Error loading model: {e}. Check your model path and internet connection.")
    sys.exit()

# --- 2. Define the Prediction Function for LIME ---
def predictor(texts):
    """Tokenizes a list of texts and returns prediction probabilities."""
    with torch.no_grad():
        inputs = tokenizer(texts,
                           return_tensors="pt",
                           padding=True,
                           truncation=True,
                           max_length=128)
        # Move inputs to the correct device
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        
        outputs = model(**inputs)
        # Return probabilities (softmax) for LIME
        probabilities = F.softmax(outputs.logits, dim=1).cpu().numpy()
    return probabilities

# --- 3. Example Data for Explanation ---
example_ai_spam_text = (
    "URGENT NOTICE: Your Retail Rewards account balance has been suspended due to "
    "an unauthorized purchase attempt on 25/09/2025. Please confirm your identity "
    "and update your details immediately via the dedicated Retail Secure Link. "
    "Failure to act within 2 hours will result in permanent loss of all accumulated "
    "Loyalty Points and account closure. Click here now: [URL]"
)
example_label_index = 2 # Target class for explanation (AI-SPAM)

# --- 4. LIME Implementation (Local Interpretability) Analysis ---
print("\n" + "="*50)
print("4. LIME (Local Interpretability) Analysis")
print("="*50)

explainer_lime = LimeTextExplainer(class_names=CLASS_NAMES)
explanation = explainer_lime.explain_instance(
    text_instance=example_ai_spam_text,
    classifier_fn=predictor,
    labels=[example_label_index],
    num_samples=5000,
    top_labels=1
)

print(f"\nLIME Explanation for Class: {CLASS_NAMES[example_label_index]}")
print("Top 10 features contributing to the AI-SPAM classification:")

feature_weights = explanation.as_list(label=example_label_index)

for feature, weight in feature_weights[:10]:
    print(f"  Token: '{feature.replace('\n', ' ')}' | Weight: {weight:.4f}")

# --- 5. Integrated Gradients (IG) Implementation ---
print("\n" + "="*50)
print("5. Integrated Gradients (IG) Analysis")
print("="*50)

# 5.1. Define the EMBEDDING attribution function for Captum
def forward_func_embeds(inputs_embeds, attention_mask=None):
    """Wrapper function for the model to be compatible with Captum."""
    # This assumes the model has a property named 'electra' for its transformer base
    # NOTE: You might need to adjust 'model.electra' based on your model architecture
    outputs = model.electra(inputs_embeds=inputs_embeds, attention_mask=attention_mask.long())
    # Pass the [CLS] token hidden state to the classification head
    logits = model.classifier(outputs[0])
    return logits

# 5.2. Prepare inputs, embeddings, and baseline (reference)
encoding = tokenizer.encode_plus(
    example_ai_spam_text,
    return_tensors='pt',
    padding='max_length',
    truncation=True,
    max_length=128
)
input_ids = encoding['input_ids'].long().to(DEVICE)
attention_mask = encoding['attention_mask'].long().to(DEVICE)

# 5.2a. Generate Input Embeddings
input_embeddings = model.electra.embeddings(input_ids=input_ids).requires_grad_()

# 5.2b. Generate Reference Embeddings (Baseline: using [PAD] token embedding)
pad_token_id = tokenizer.pad_token_id
pad_embedding = model.electra.embeddings.word_embeddings(torch.tensor(pad_token_id).to(DEVICE))
reference_embeddings = pad_embedding.unsqueeze(0).expand(input_embeddings.shape)

# 5.3. Initialize and Compute IG
ig = IntegratedGradients(forward_func_embeds)
print("Calculating Integrated Gradients (Targeting Embeddings)...")

attributions_ig, delta = ig.attribute(
    inputs=input_embeddings,
    baselines=reference_embeddings,
    target=example_label_index,
    n_steps=50,
    additional_forward_args=(attention_mask,),
    return_convergence_delta=True
)

# 5.4. Process and Display Numerical Results
all_tokens = tokenizer.convert_ids_to_tokens(input_ids.flatten())
# Sum attributions across the embedding dimension (get one score per token)
attributions_ig_sum = attributions_ig.sum(dim=-1).squeeze(0)
attributions_ig_sum_numpy = attributions_ig_sum.cpu().detach().numpy()

# Combine tokens and IG scores for text-only output
ig_results = []
for token, score in zip(all_tokens, attributions_ig_sum_numpy):
    # Filter out special tokens and padded tokens for cleaner output
    if token not in ['[CLS]', '[SEP]', '[PAD]']:
        ig_results.append((token, score))

# Sort by absolute score to find the most impactful words
ig_results.sort(key=lambda x: abs(x[1]), reverse=True)

print(f"\nIntegrated Gradients Explanation for Class: {CLASS_NAMES[example_label_index]}")
print("Top 10 tokens ranked by absolute IG score (Impact):")
for token, score in ig_results[:10]:
    # Replace subword prefix for printing
    print(f"  Token: '{token.replace('##', '')}' | IG Score (Logit): {score:.4f}")

# 5.5. Generate and Display Visual Heatmap
print("\n" + "="*50)
print("6. Visual Heatmap Generation (IG)")
print("="*50)

# Get the indices of non-special tokens
token_list = input_ids.flatten().cpu().numpy().tolist()
# Find the end of the text (before [SEP])
try:
    end_index = token_list.index(tokenizer.sep_token_id)
except ValueError:
    end_index = len(token_list) # Should not happen if max_length is large enough

# Extract relevant tokens and scores (Skip [CLS] at index 0)
visual_tokens = all_tokens[1:end_index]
visual_scores = attributions_ig_sum.tolist()[1:end_index]


# Save the visualization to an HTML file
html_file_path = plot_text_heatmap(visual_tokens, visual_scores)

if html_file_path:
    print(f"\n[NEXT STEP] Open this file in your browser to view the heatmap: {html_file_path}")

print("\n--- XAI Analysis Complete ---")

--- Initializing Model and Tokenizer ---
Model and Tokenizer loaded successfully.

--- 4. LIME (Local Interpretability) Analysis ---

LIME Explanation for Class: AI-SPAM
Top 10 features contributing to the AI-SPAM classification:
  Token: 'unauthorized' | Weight: -0.0011
  Token: 'confirm' | Weight: -0.0011
  Token: 'and' | Weight: -0.0011
  Token: 'Click' | Weight: -0.0010
  Token: '25' | Weight: -0.0010
  Token: 'identity' | Weight: -0.0009
  Token: 'to' | Weight: -0.0009
  Token: 'Rewards' | Weight: 0.0007
  Token: 'attempt' | Weight: 0.0004
  Token: '2' | Weight: 0.0004

--- 5. Integrated Gradients (IG) Analysis ---
Calculating Integrated Gradients (Targeting Embeddings)...

Integrated Gradients Explanation for Class: AI-SPAM
Top 10 tokens ranked by absolute IG score (Impact):
  Token: '.' | IG Score (Logit): 0.0046
  Token: '.' | IG Score (Logit): 0.0046
  Token: 'been' | IG Score (Logit): 0.0044
  Token: '.' | IG Score (Logit): 0.0041
  Token: 'to' | IG Score (Logit): 0.0039
  To


--- XAI Analysis Complete ---
